In [11]:
from dotenv import load_dotenv

load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore


PDF/문서 로드
→ 긴 문서를 작은 chunk로 자름
→ 각 chunk를 embedding 숫자로 변환
→ vector store에 저장
→ 질문과 비슷한 chunk 검색

In [ ]:
from langchain_core.documents import Document

sample_docs = [
    Document(page_content="""
    KLA는 반도체 제조 과정에서 발생하는 결함을 검사·측정하고 데이터를 분석하는 공정 제어 솔루션을 제공합니다.
    AI, HBM, 첨단 공정과 패키징 기술의 발전으로 반도체 구조와 제조 과정이 복잡해지면서 정밀한 검사·계측의 중요성이 커지고 있습니다.
    KLA는 이러한 기술을 통해 고객의 수율 향상과 생산 비용 절감을 지원하며 관련 시장의 수요 증가에 대응하고 있습니다.
    """
    )
]

test_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 20
)

print(test_splitter.split_documents(sample_docs))

[Document(metadata={}, page_content='KLA는 반도체 제조 과정에서 발생하는 결함을 검사·측정하고 데이터를 분석하는 공정 제어 솔루션을 제공합니다.'), Document(metadata={}, page_content='AI, HBM, 첨단 공정과 패키징 기술의 발전으로 반도체 구조와 제조 과정이 복잡해지면서 정밀한 검사·계측의 중요성이 커지고 있습니다.'), Document(metadata={}, page_content='KLA는 이러한 기술을 통해 고객의 수율 향상과 생산 비용 절감을 지원하며 관련 시장의 수요 증가에 대응하고 있습니다.')]


In [14]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, chunk_overlap=200, add_start_index=True
)

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')


In [ ]:

loader = PyPDFLoader('/10k-research-agent/data/row/KLA-10-K-2026.pdf')
docs = loader.load()

# 위에 pdf 로딩한거 잘라서 청크로 만들기
chunks = text_splitter.split_documents(docs)

print('원본 문서 페이지 수를 출력해보자',len(docs))
print('잘 분리됬는지 청크 수', len(chunks))
#벡터 스토어에 저장해보기
kla_vectorstore = InMemoryVectorStore(embedding=embeddings)

ids = kla_vectorstore.add_documents(chunks)

# 저장이 잘되었는지 출력해보기
# 반복문을 돌려서 출력해보자 -> 청크 번호를 카운팅 해보자
# 근데 i 로 넣어서 하면 되겠지 했지만 왠걸 오류가 발생한다
#: too many values to unpack (expected 2) 발생한다 2개인자값을 받아야 하는데 i, doc_id
# 실제 들어오는 인자는 문자열1개란 말이죠... enumerate 괜찮다 함 아니며 store.items로 호출한다

test_ids = list(kla_vectorstore.store.keys())

for i,doc_id in enumerate(test_ids[:3]):
    item = kla_vectorstore.store[doc_id]
    
    print("=" * 20)
    print("청크 번호:", i)
    print("청크 ID:", doc_id)
    print("페이지:", item["metadata"]["page"] + 1)
    print("시작 위치:", item["metadata"]["start_index"])
    print("글자 수:", len(item["text"]))
    print("=" * 20)


원본 문서 페이지 수를 출력해보자 111
잘 분리됬는지 청크 수 614
청크 번호: 0
청크 ID: da19bfba-9b66-4dcb-b58e-a251f3d3f464
페이지: 1
시작 위치: 0
글자 수: 995
청크 번호: 1
청크 ID: 959bbaa6-6fe7-433e-95a7-f65fea377bdf
페이지: 1
시작 위치: 841
글자 수: 914
청크 번호: 2
청크 ID: d2ac3e58-7d77-432f-a36d-0949ff216be6
페이지: 1
시작 위치: 1558
글자 수: 996


In [ ]:
from langchain_core.tools import tool

@tool
def search_kla_10k(query: str):
    pass